# 🚗 Sistema Predictivo de Accidentes — Versión Final
## Universidad Tecnológica de Panamá — Maestría en Analítica de Datos
### Proyecto Integrador 2026

**Mejoras implementadas sobre revisión técnica:**
- ✅ Fix Data Leakage: IQR aplicado **solo** sobre train set
- ✅ Modelo de Ocurrencia: **Regresión de Poisson** (estándar actuarial)
- ✅ Variable target unificada: `Target_Severity` (0/1/2) desde MUTCD
- ✅ Preprocesamiento con **ColumnTransformer** + `OrdinalEncoder`
- ✅ Pipeline completamente serializable con `joblib`
- ✅ **CalibratedClassifierCV** para probabilidades calibradas
- ✅ Métricas comerciales: Lift Curve, Gini Coefficient

---
## 1. Setup & Librerías

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, accuracy_score
)
import joblib
import statsmodels.api as sm

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


---
## 2. Carga de Datos

In [2]:
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/UTP/2024/s107_Proyector_Integrador_1/final_project/notebooks/data/US_Accidents_FL.csv'
df_raw = pd.read_csv(CSV_PATH)
print(f"Dataset: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
df_raw.head(3)

Mounted at /content/drive
Dataset: 880,192 filas × 46 columnas


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-116062,Source2,3,2016-11-30 15:36:03,2016-11-30 17:09:22,27.981,-82.327,NaN,NaN,0.010,Queueing traffic and two left lane blocked due to accident on I-75 Southbound after Exits 260 260A 260B FL-574 Dr Martin Luther King Jr Blvd.,E Dr Martin Luther King Jr Blvd,Tampa,Hillsborough,FL,33610,US,US/Eastern,KVDF,2016-11-30 15:35:00,80.600,NaN,70.000,29.940,10.000,SSW,5.800,NaN,Overcast,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day
1,A-116063,Source2,3,2016-11-30 16:25:35,2016-11-30 17:12:25,27.981,-82.327,NaN,NaN,0.010,Queueing traffic and 2 left lane closed due to accident on I-75 Southbound at Exits 260 260A 260B FL-574 Dr Martin Luther King Jr Blvd.,E Dr Martin Luther King Jr Blvd,Tampa,Hillsborough,FL,33610,US,US/Eastern,KVDF,2016-11-30 16:35:00,80.600,NaN,65.000,29.940,10.000,WSW,6.900,NaN,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day
2,A-116064,Source2,2,2016-11-30 16:40:31,2016-11-30 17:10:19,25.628,-80.374,NaN,NaN,0.010,Accident on FL-992 152nd St at Lincoln Blvd.,SW 152nd St,Miami,Miami-Dade,FL,33157-1147,US,US/Eastern,KTMB,2016-11-30 16:53:00,80.100,NaN,71.000,29.960,10.000,SE,9.200,NaN,Mostly Cloudy,False,False,True,False,False,False,False,False,False,True,False,True,False,Day,Day,Day,Day


---
## 3. Limpieza & Ingeniería de Features

In [3]:
# Imputación básica sobre dataset RAW (antes del split)
num_cols = df_raw.select_dtypes(include=np.number).columns.tolist()
cat_cols = df_raw.select_dtypes(include='object').columns.tolist()

for col in num_cols:
    df_raw[col] = df_raw[col].fillna(df_raw[col].median())
for col in cat_cols:
    df_raw[col] = df_raw[col].fillna(df_raw[col].mode()[0])

print(f"Nulos restantes: {df_raw.isnull().sum().sum()}")

Nulos restantes: 0


In [4]:
# Variables temporales
df_raw['Start_Time']   = pd.to_datetime(df_raw['Start_Time'], errors='coerce')
df_raw['End_Time']     = pd.to_datetime(df_raw['End_Time'],   errors='coerce')
df_raw['Hour']         = df_raw['Start_Time'].dt.hour
df_raw['DayOfWeek']    = df_raw['Start_Time'].dt.dayofweek
df_raw['Month']        = df_raw['Start_Time'].dt.month
df_raw['Duration_min'] = (df_raw['End_Time'] - df_raw['Start_Time']).dt.total_seconds() / 60

# Clasificación MUTCD (estándar FHWA)
def mutcd_category(dur):
    if pd.isna(dur) or dur < 30: return 'Menor'
    elif dur <= 120:              return 'Intermedio'
    else:                         return 'Mayor'

df_raw['MUTCD_Category'] = df_raw['Duration_min'].apply(mutcd_category)

# Target unificado para Etapa 2
cat_map = {'Menor': 0, 'Intermedio': 1, 'Mayor': 2}
df_raw['Target_Severity'] = df_raw['MUTCD_Category'].map(cat_map)

# Redondeo geográfico (~1 km²) para Etapa 1
df_raw['lat_round'] = df_raw['Start_Lat'].round(2)
df_raw['lng_round'] = df_raw['Start_Lng'].round(2)

print("Distribución Target_Severity (0=Menor, 1=Intermedio, 2=Mayor):")
print(df_raw['Target_Severity'].value_counts())

Distribución Target_Severity (0=Menor, 1=Intermedio, 2=Mayor):
Target_Severity
1    348150
0    269108
2    262934
Name: count, dtype: int64


---
## 4. Train/Test Split — ANTES del escalado y outlier removal
> **Fix Revisión Técnica #1:** Los bounds del IQR se calculan **solo** sobre
> `X_train` para evitar data leakage hacia el test set.

In [5]:
FEATURE_COLS = [
    'Start_Lat', 'Start_Lng', 'City', 'County',
    'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction',
    'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
    'Traffic_Calming', 'Traffic_Signal',
    'Temperature(F)', 'Humidity(%)', 'Visibility(mi)',
    'Wind_Speed(mph)', 'Precipitation(in)',
    'Weather_Condition', 'Sunrise_Sunset',
    'Hour', 'DayOfWeek', 'Month'
]
TARGET_COL = 'Target_Severity'

df_model = df_raw[FEATURE_COLS + [TARGET_COL]].dropna()
X = df_model[FEATURE_COLS]
y = df_model[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Train: 612,500  |  Test: 153,125


In [6]:
# IQR calculado SOLO en train — bounds aplicados luego a test
num_features_iqr = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()

train_bounds = {}
for feat in num_features_iqr:
    Q1, Q3 = X_train[feat].quantile(0.25), X_train[feat].quantile(0.75)
    IQR = Q3 - Q1
    train_bounds[feat] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

mask_train = pd.Series(True, index=X_train.index)
mask_test  = pd.Series(True, index=X_test.index)
for feat, (lo, hi) in train_bounds.items():
    mask_train &= X_train[feat].between(lo, hi)
    mask_test  &= X_test[feat].between(lo, hi)

X_train_c = X_train[mask_train].copy()
y_train_c = y_train[mask_train].copy()
X_test_c  = X_test[mask_test].copy()
y_test_c  = y_test[mask_test].copy()

print(f"Tras IQR → Train: {len(X_train_c):,}  |  Test: {len(X_test_c):,}")

Tras IQR → Train: 493,521  |  Test: 123,547


---
## 5. Preprocesamiento — ColumnTransformer
> **Fix #4:** `OrdinalEncoder` con manejo de valores desconocidos en lugar del ciclo `LabelEncoder`.

In [8]:
num_cols_pp = X_train_c.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols_pp = X_train_c.select_dtypes(include=['object', 'bool']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_pp),
        ('cat', OrdinalEncoder(
            handle_unknown='use_encoded_value', unknown_value=-1
        ), cat_cols_pp)
    ],
    remainder='passthrough'
)
print(f"Numéricas: {len(num_cols_pp)}  |  Categóricas: {len(cat_cols_pp)}")

Numéricas: 10  |  Categóricas: 16


---
## 6. Etapa 1 — Modelo de Ocurrencia (Regresión de Poisson)
> **Fix #2:** Se eliminan los ceros artificiales. Se usa Poisson GLM sobre frecuencias reales por zona geográfica — estándar actuarial para frecuencia de siniestros.

In [9]:
# Tabla de frecuencias reales por zona-hora
freq_df = (df_raw
    .groupby(['lat_round', 'lng_round', 'Hour', 'Month', 'County'])
    .agg(
        accidentes      = ('Target_Severity', 'count'),
        temp_mean       = ('Temperature(F)', 'mean'),
        humidity_mean   = ('Humidity(%)', 'mean'),
        visibility_mean = ('Visibility(mi)', 'mean'),
        rain_mean       = ('Precipitation(in)', 'mean'),
    )
    .reset_index()
)
print(f"Zonas-hora únicas: {freq_df.shape[0]:,}")
print(f"Frecuencia media por zona-hora: {freq_df['accidentes'].mean():.2f}")
freq_df.head()

Zonas-hora únicas: 405,203
Frecuencia media por zona-hora: 1.89


,lat_round,lng_round,Hour,Month,County,accidentes,temp_mean,humidity_mean,visibility_mean,rain_mean
0,24.550,-81.780,13.000,8.000,Monroe,1,88.000,65.000,10.000,0.000
1,24.560,-81.810,12.000,8.000,Monroe,1,88.000,67.000,10.000,0.000
2,24.560,-81.800,15.000,5.000,Monroe,1,87.000,65.000,10.000,0.000
3,24.560,-81.790,16.000,5.000,Monroe,1,82.000,69.000,10.000,0.000
4,24.560,-81.780,14.000,4.000,Monroe,1,83.000,53.000,10.000,0.000


In [10]:
# Codificar County para Poisson
county_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
freq_df['county_enc'] = county_encoder.fit_transform(freq_df[['County']])

poisson_features = ['temp_mean', 'humidity_mean', 'visibility_mean',
                    'rain_mean', 'Hour', 'Month', 'county_enc']

X_pois = sm.add_constant(freq_df[poisson_features])
y_pois = freq_df['accidentes']

poisson_model = sm.GLM(y_pois, X_pois, family=sm.families.Poisson()).fit()
print(poisson_model.summary())

freq_df['freq_predicha'] = poisson_model.predict(X_pois)
mae = np.abs(freq_df['accidentes'] - freq_df['freq_predicha']).mean()
print(f"\nMAE Poisson: {mae:.3f} accidentes/zona-hora")

                 Generalized Linear Model Regression Results                  
Dep. Variable:             accidentes   No. Observations:               405203
Model:                            GLM   Df Residuals:                   405195
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -7.6374e+05
Date:                Sun, 10 May 2026   Deviance:                   5.8410e+05
Time:                        00:29:46   Pearson chi2:                 1.28e+06
No. Iterations:                     7   Pseudo R-squ. (CS):            0.02577
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.4081      0.014     

In [11]:
# Visualización: Frecuencia observada vs predicha
fig = px.scatter(
    freq_df.sample(5000, random_state=42),
    x='accidentes', y='freq_predicha',
    opacity=0.4, color_continuous_scale='YlOrRd',
    template='plotly_white',
    title='Frecuencia Observada vs Predicha — Modelo Poisson',
    labels={'accidentes': 'Accidentes Observados', 'freq_predicha': 'Frecuencia Predicha (Poisson)'}
)
fig.add_shape(type='line', x0=0, x1=freq_df['accidentes'].max(),
              y0=0, y1=freq_df['accidentes'].max(),
              line=dict(color='red', dash='dash'))
fig.update_layout(height=500, width=700)
fig.show()

---
## 7. Etapa 2 — Modelo de Severidad (Pipeline Serializable)
> **Fix #4 & #5:** `ColumnTransformer` + `Pipeline` de sklearn. **Fix #6:** `CalibratedClassifierCV`.

In [12]:
# Undersampling balanceado
df_bal = pd.concat([X_train_c, y_train_c], axis=1)
min_count = df_bal['Target_Severity'].value_counts().min()
df_balanced = (df_bal
    .groupby('Target_Severity', group_keys=False)
    .apply(lambda g: g.sample(n=min_count, random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
X_tr = df_balanced[FEATURE_COLS]
y_tr = df_balanced['Target_Severity']
print(f"Distribución balanceada:\n{y_tr.value_counts()}")
print(f"Total registros de entrenamiento: {len(y_tr):,}")

Distribución balanceada:
Target_Severity
0    99030
2    99030
1    99030
Name: count, dtype: int64
Total registros de entrenamiento: 297,090


In [13]:
# Pipeline principal — Fix #5: completamente serializable
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', CalibratedClassifierCV(
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        cv=5, method='sigmoid'   # Platt scaling → probabilidades calibradas
    ))
])

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline_rf, X_tr, y_tr, cv=kf, scoring='accuracy')
print(f"CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Scores por fold: {cv_scores.round(4)}")

CV Accuracy: 0.6007 ± 0.0016
Scores por fold: [0.601  0.6016 0.5985 0.5995 0.603 ]


In [14]:
# Entrenamiento y evaluación
pipeline_rf.fit(X_tr, y_tr)

y_pred = pipeline_rf.predict(X_test_c)
y_prob = pipeline_rf.predict_proba(X_test_c)

LABELS = ['Menor (0)', 'Intermedio (1)', 'Mayor (2)']
print("=" * 60)
print("EVALUACIÓN EN TEST SET")
print("=" * 60)
print(classification_report(y_test_c, y_pred, target_names=LABELS))
print(f"ROC AUC (OvR, weighted): {roc_auc_score(y_test_c, y_prob, multi_class='ovr', average='weighted'):.4f}")
print(f"Accuracy:                {accuracy_score(y_test_c, y_pred):.4f}")

EVALUACIÓN EN TEST SET
                precision    recall  f1-score   support

     Menor (0)       0.42      0.61      0.50     24788
Intermedio (1)       0.66      0.57      0.61     54314
     Mayor (2)       0.72      0.66      0.69     44445

      accuracy                           0.61    123547
     macro avg       0.60      0.61      0.60    123547
  weighted avg       0.63      0.61      0.62    123547

ROC AUC (OvR, weighted): 0.7936
Accuracy:                0.6091


In [15]:
# Matriz de confusión
cm = confusion_matrix(y_test_c, y_pred)
fig = px.imshow(cm,
    labels=dict(x="Predicho", y="Real", color="Conteo"),
    x=LABELS, y=LABELS,
    text_auto=True, color_continuous_scale='Blues',
    title='Matriz de Confusión — Modelo de Severidad'
)
fig.update_layout(height=450, width=550, template='plotly_white')
fig.show()

---
## 8. Métricas Comerciales
> **Fix #7:** Métricas que hablan el idioma actuarial.

In [16]:
# ── Curva Lift — Clase Mayor (accidentes de alto impacto) ────────────────────
prob_mayor   = y_prob[:, 2]
actual_mayor = (np.array(y_test_c) == 2).astype(int)

order = np.argsort(-prob_mayor)
actual_sorted = actual_mayor[order]
n = len(actual_sorted)

cumpos = np.cumsum(actual_sorted)
baseline = actual_mayor.mean()
lift    = cumpos / (np.arange(1, n + 1) * baseline)
pct_pop = np.arange(1, n + 1) / n * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=pct_pop, y=lift, mode='lines',
    line=dict(color='#e63946', width=2.5), name='Modelo'))
fig.add_hline(y=1, line_dash='dash', line_color='gray',
    annotation_text='Línea base (random)')
fig.update_layout(
    title='Curva Lift — Clase "Mayor" (alto impacto en tráfico)',
    xaxis_title='% de Pólizas Inspeccionadas',
    yaxis_title='Lift', template='plotly_white', height=450, width=850
)
fig.show()

print("=== LIFT POR PERCENTIL ===")
for pct in [10, 20, 30, 50]:
    idx = min(int(n * pct / 100), n - 1)
    print(f"Top {pct:2d}%: Lift {lift[idx]:.2f}x — "
          f"detecta {lift[idx]:.1f}x más accidentes 'Mayor' que al azar")

Output hidden; open in https://colab.research.google.com to view.

In [17]:
# ── Gini Coefficient ─────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score
gini = 2 * roc_auc_score(actual_mayor, prob_mayor) - 1
print(f"Gini Coefficient (clase Mayor): {gini:.4f}")
print(f"AUC (clase Mayor):              {(gini + 1) / 2:.4f}")
print()
if gini >= 0.60:
    nivel = "🟢 BUENO — adecuado para selección de riesgo"
elif gini >= 0.40:
    nivel = "🟡 ACEPTABLE — útil para tarifación"
else:
    nivel = "🔴 BAJO — revisar features"
print(f"Nivel del modelo: {nivel}")

Gini Coefficient (clase Mayor): 0.6710
AUC (clase Mayor):              0.8355

Nivel del modelo: 🟢 BUENO — adecuado para selección de riesgo


In [18]:
# Resumen actuarial
print("""
=============================================================
     RESUMEN ACTUARIAL DEL SISTEMA PREDICTIVO
=============================================================

  ETAPA 1 (Poisson GLM)  -> E[Frecuencia]
  * Predice # esperado de siniestros por zona/mes
  * Salida usada en: Prima = E[Freq] x E[Costo] x loading

  ETAPA 2 (RF + Calibracion) -> P(Severidad | siniestro)
  * Probabilidades calibradas para 3 niveles MUTCD
  * Directamente usable en factor de agravacion

  PROXIMOS PASOS:
  1. Dataset de costos para calibrar E[Costo_Siniestro]
  2. Mapeo a variables permitidas SBP (Panama)
  3. Dashboard Plotly Dash / Streamlit para demo comercial
=============================================================
""")


     RESUMEN ACTUARIAL DEL SISTEMA PREDICTIVO

  ETAPA 1 (Poisson GLM)  -> E[Frecuencia]
  * Predice # esperado de siniestros por zona/mes
  * Salida usada en: Prima = E[Freq] x E[Costo] x loading

  ETAPA 2 (RF + Calibracion) -> P(Severidad | siniestro)
  * Probabilidades calibradas para 3 niveles MUTCD
  * Directamente usable en factor de agravacion

  PROXIMOS PASOS:
  1. Dataset de costos para calibrar E[Costo_Siniestro]
  2. Mapeo a variables permitidas SBP (Panama)
  3. Dashboard Plotly Dash / Streamlit para demo comercial



---
## 9. Sistema de Predicción Encapsulado (API-Ready)
> **Fix #5:** Clase sin dependencias de variables globales — serializable con `joblib`.

In [19]:
class AccidentPredictionSystem:
    """
    Sistema predictivo encapsulado.
    Etapa 1: Poisson (ocurrencia)
    Etapa 2: RandomForest Calibrado (severidad)
    100% serializable con joblib.
    """

    def __init__(self, severity_pipeline, poisson_model,
                 county_encoder, iqr_bounds, feature_cols):
        self.severity_pipeline = severity_pipeline
        self.poisson_model     = poisson_model
        self.county_encoder    = county_encoder
        self.iqr_bounds        = iqr_bounds
        self.feature_cols      = feature_cols

    def _clip_outliers(self, X: pd.DataFrame) -> pd.DataFrame:
        X_c = X.copy()
        for feat, (lo, hi) in self.iqr_bounds.items():
            if feat in X_c.columns:
                X_c[feat] = X_c[feat].clip(lo, hi)
        return X_c

    def predict_severity(self, X: pd.DataFrame) -> pd.DataFrame:
        X_c   = self._clip_outliers(X[self.feature_cols])
        preds = self.severity_pipeline.predict(X_c)
        proba = self.severity_pipeline.predict_proba(X_c)
        labels = {0: 'Menor', 1: 'Intermedio', 2: 'Mayor'}
        return pd.DataFrame({
            'clase':           [labels[p] for p in preds],
            'prob_menor':      proba[:, 0].round(4),
            'prob_intermedio': proba[:, 1].round(4),
            'prob_mayor':      proba[:, 2].round(4),
        })

    def predict_frequency(self, zone_df: pd.DataFrame) -> np.ndarray:
        df = zone_df.copy()
        df['county_enc'] = self.county_encoder.transform(df[['County']])
        feats = ['temp_mean', 'humidity_mean', 'visibility_mean',
                 'rain_mean', 'Hour', 'Month', 'county_enc']
        X_p = sm.add_constant(df[feats], has_constant='add')
        return self.poisson_model.predict(X_p).values

    def save(self, path: str):
        joblib.dump(self, path)
        print(f"✅ Sistema guardado: {path}")

    @classmethod
    def load(cls, path: str):
        return joblib.load(path)


# Instanciar y guardar
OUTPUT_DIR = '/content/drive/MyDrive/UTP/2024/s107_Proyector_Integrador_1/final_project/notebooks/proyecto_integrador_3'
system = AccidentPredictionSystem(
    severity_pipeline = pipeline_rf,
    poisson_model     = poisson_model,
    county_encoder    = county_encoder,
    iqr_bounds        = train_bounds,
    feature_cols      = FEATURE_COLS
)
system.save(f'{OUTPUT_DIR}/accident_prediction_system.joblib')

✅ Sistema guardado: /content/drive/MyDrive/UTP/2024/s107_Proyector_Integrador_1/final_project/notebooks/proyecto_integrador_3/accident_prediction_system.joblib


In [4]:
# ── Test de carga y predicción ────────────────────────────────────────────────
sys_loaded = AccidentPredictionSystem.load(
    f'{OUTPUT_DIR}/accident_prediction_system.joblib'
)

sample = X_test_c.head(10)
result = sys_loaded.predict_severity(sample)
print("Predicciones de muestra (primeras 10 filas):")
print(result.to_string(index=False))

NameError: name 'AccidentPredictionSystem' is not defined

---
## ✅ Resumen de Correcciones Aplicadas

| Issue | Fix Implementado |
|---|---|
| IQR sobre todo el dataset (data leakage) | IQR calculado solo en `X_train` → bounds aplicados a `X_test` |
| Ceros artificiales en modelo de ocurrencia | `Poisson GLM` (statsmodels) sobre frecuencias reales por zona |
| Targets divergentes entre notebooks | `Target_Severity` unificado (0/1/2) desde `MUTCD_Category` |
| `LabelEncoder` en ciclo (anti-patrón) | `ColumnTransformer` + `OrdinalEncoder(handle_unknown=...)` |
| `AccidentPredictionSystem` no serializable | Clase encapsula todo el `Pipeline` → `joblib`-ready |
| Sin probabilidades calibradas | `CalibratedClassifierCV` con Platt scaling |
| Sin métricas de negocio | Lift Curve + Gini Coefficient + Resumen actuarial |

## 10. Exportar resultados
